# 알약캡쳐 — 알약 검출 모델 학습 (YOLOv8n, Colab 무료 GPU)

순서: 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후, 메뉴 **런타임 → 모두 실행**.

준비물: Google Drive의 `pillcapture` 폴더에 Roboflow에서 받은 zip 2개를 넣어 두세요.
- `CountingPills.v36i.yolov8.zip`
- `Pill Detection.v22i.yolov8.zip`

(없으면 아래 셀이 업로드 창을 띄웁니다.) 학습 데이터 출처: Pill-Counter(TetraHikari), CountingPills·Pill Detection(Roboflow Universe, CC BY 4.0).

In [ ]:
!nvidia-smi -L || echo 'GPU 없음: 런타임 유형을 T4 GPU 로 바꿔 주세요'
!pip -q install ultralytics==8.3.* onnx onnxruntime 2>/dev/null | tail -1
import ultralytics; print('ultralytics', ultralytics.__version__)

In [ ]:
# 데이터 zip 찾기 (Google Drive → 없으면 업로드)
import glob, os
from google.colab import drive
drive.mount('/content/drive')
find = lambda pat: (glob.glob(f'/content/drive/MyDrive/pillcapture/{pat}') + glob.glob(f'/content/drive/MyDrive/{pat}') + glob.glob(f'/content/{pat}'))
CP = find('CountingPills*.zip'); KR = find('Pill Detection*.zip')
if not CP or not KR:
    from google.colab import files
    print('Roboflow zip 2개를 선택해 주세요'); up = files.upload()
    CP = find('CountingPills*.zip'); KR = find('Pill Detection*.zip')
CP, KR = CP[0], KR[0]; print('CP', CP); print('KR', KR)

In [ ]:
# 학습 스크립트·사용자 라벨 받기 + 데이터 합치기
!rm -rf /content/repo && git clone -q --depth 1 https://github.com/728haven-oss/pill-capture /content/repo
!python /content/repo/ml/colab/prep_data.py /content/work "$CP" "$KR" /content/repo/ml/colab/user_lab
!cat /content/work/data/data.yaml

In [ ]:
# 학습 (YOLOv8n, 640px). T4 기준 40 epoch ≈ 40~60분
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(data='/content/work/data/data.yaml', epochs=40, imgsz=640, batch=32, patience=12, project='/content/runs', name='pill', exist_ok=True,
            hsv_h=0.1, hsv_s=0.6, hsv_v=0.4, degrees=15, scale=0.5, fliplr=0.5, flipud=0.5, mosaic=1.0, close_mosaic=8, workers=4, plots=False)
print('best:', '/content/runs/pill/weights/best.pt')

In [ ]:
# 개수 정확도 평가 (테스트셋, 신뢰도 임계값별)
import glob, numpy as np
from ultralytics import YOLO
m = YOLO('/content/runs/pill/weights/best.pt')
imgs = sorted(glob.glob('/content/work/data/test/images/*.jpg'))
gt = {p: sum(1 for l in open(p.replace('/images/','/labels/')[:-4]+'.txt') if l.strip()) for p in imgs}
best = None
for conf in [0.25, 0.35, 0.45, 0.55, 0.65]:
    err = []
    for i in range(0, len(imgs), 32):
        for p, r in zip(imgs[i:i+32], m.predict(imgs[i:i+32], conf=conf, iou=0.5, imgsz=640, verbose=False, max_det=1000)):
            err.append(abs(len(r.boxes) - gt[p]))
    err = np.array(err); ex = (err == 0).mean(); print(f'conf {conf}: MAE {err.mean():.2f}  정확 {100*ex:.1f}%  ±1 {100*(err<=1).mean():.1f}%')
    if best is None or err.mean() < best[1]: best = (conf, err.mean())
print('추천 임계값', best[0])

In [ ]:
# ONNX 내보내기 (앱 내장용: onnxruntime-web) → Google Drive 저장 + 다운로드
from ultralytics import YOLO
import shutil, os, json
m = YOLO('/content/runs/pill/weights/best.pt')
path = m.export(format='onnx', imgsz=640, opset=12, simplify=True, dynamic=False, half=False)
os.makedirs('/content/drive/MyDrive/pillcapture', exist_ok=True)
shutil.copy(path, '/content/drive/MyDrive/pillcapture/pillyolo.onnx'); shutil.copy('/content/runs/pill/weights/best.pt', '/content/drive/MyDrive/pillcapture/pillyolo_best.pt')
json.dump({'conf': best[0], 'imgsz': 640}, open('/content/drive/MyDrive/pillcapture/pillyolo.json', 'w'))
print('저장:', os.path.getsize(path)//1024, 'KB → Drive/pillcapture/pillyolo.onnx')
from google.colab import files; files.download(path)